In [8]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
import category_encoders as ce
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

from catboost import CatBoostClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, 
    recall_score, f1_score, make_scorer, confusion_matrix, classification_report)


import warnings
warnings.filterwarnings('ignore')

In [9]:
DATA_PATH = "/kaggle/input/datasets/omarhusnye/electronicsnew/clean_events_electronics_computers.csv"
df = pd.read_csv(DATA_PATH)

df.head(10)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,year,...,category_l1,category_l2,category_l3,is_pc_component,component_type,product_group,is_likely_bot_session,price_band,brand_is_unknown,category_l3_is_unknown
0,2020-09-24 11:57:06+00:00,view,1996170,2144415922528452715,electronics.telephone,unknown,31.90,1515915625519388267,LJuJVLEjPT,2020,...,electronics,telephone,unknown,False,not_a_component,Mobile & Telephony,False,$25-50,1,1
1,2020-09-24 11:57:26+00:00,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY,2020,...,computers,components,cooler,True,cooler,PC Components,False,<$25,0,0
2,2020-09-24 11:57:33+00:00,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08,2020,...,computers,peripherals,printer,False,not_a_component,PC Peripherals,False,$100-250,0,0
3,2020-09-24 11:58:23+00:00,view,3791349,2144415935086199225,computers.desktop,unknown,215.41,1515915625519388877,J1t6sIYXiV,2020,...,computers,desktop,unknown,False,not_a_component,PC Systems,False,$100-250,1,1
4,2020-09-24 11:58:24+00:00,view,716611,2144415923694469257,computers.network.router,d-link,53.14,1515915625519388882,kVBeYDPcBw,2020,...,computers,network,router,False,not_a_component,PC Peripherals,False,$50-100,0,0
5,2020-09-24 11:58:31+00:00,view,716611,2144415923694469257,computers.network.router,d-link,53.14,1515915625519388929,F3VB9LYp39,2020,...,computers,network,router,False,not_a_component,PC Peripherals,False,$50-100,0,0
6,2020-09-24 12:00:00+00:00,view,1080093,2144415923107266682,computers.peripherals.printer,ricoh,268.17,1515915625519389483,63xjTFC54g,2020,...,computers,peripherals,printer,False,not_a_component,PC Peripherals,False,$250-500,0,0
7,2020-09-24 12:00:01+00:00,view,1455459,2144415927049912542,electronics.video.tv,sony,635.63,1515915625519385419,sF2S2yMO09,2020,...,electronics,video,tv,False,not_a_component,Video & Cameras,False,$500-1k,0,0
8,2020-09-24 12:00:33+00:00,view,523117,2144415924491387038,computers.components.motherboard,asrock,73.81,1515915625519334445,HycmCUvnFr,2020,...,computers,components,motherboard,True,motherboard,PC Components,False,$50-100,0,0
9,2020-09-24 12:00:37+00:00,view,10914,2144415925053423789,electronics.camera.video,sony,40.95,1515915625519389726,kYKAorW97d,2020,...,electronics,camera,video,False,not_a_component,Video & Cameras,False,$25-50,0,0


In [10]:
# Ensure event_time is parsed safely to datetime (UTC-aware)
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

In [39]:
# Structural & Overview Inspection

print("=" * 60)
print("DATASET OVERVIEW & STRUCTURAL INSPECTION")
print("=" * 60)
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

print(f"Date Range: {df['event_time'].min()} to {df['event_time'].max()}")

print(f"Unique Users: {df['user_id'].nunique():,}")

print(f"Unique Sessions: {df['user_session'].nunique():,}")

print(f"Unique Products: {df['product_id'].nunique():,}")

print(f"Unique Categories (L3): {df['category_id'].nunique():,}")

print(f"Unique Brands: {df['brand'].nunique():,}")

DATASET OVERVIEW & STRUCTURAL INSPECTION
Shape: 485,370 rows, 28 columns
Date Range: 2020-09-24 11:57:06+00:00 to 2021-02-28 23:58:14+00:00
Unique Users: 204,049
Unique Sessions: 252,064
Unique Products: 23,797
Unique Categories (L3): 137
Unique Brands: 484


In [40]:
# Sort Chronologically
df.sort_values(by=['user_session', 'event_time'], ascending=[True, True], inplace=True)
df.reset_index(drop=True, inplace=True)

In [41]:
# Data Validation & Integrity Filters

print("\n" + "=" * 60)
print("EXECUTING SANITY CHECKS & FILTERS")
print("=" * 60)

# Check: Prices
non_pos_prices = df[df['price'] <= 0]
print(f"[Check 1] Non-positive prices (<= 0) dropped: {len(non_pos_prices):,} rows")
df = df[df['price'] > 0]

# Check: Duplicates
dup_events_mask = df.duplicated(subset=['user_session', 'event_time', 'event_type', 'product_id'])
print(f"[Check 2] Duplicate identical events dropped: {dup_events_mask.sum():,} rows")
df = df[~dup_events_mask]

# Check: Bots
if 'is_likely_bot_session' in df.columns:
    bot_sessions = df[df['is_likely_bot_session'] == True]['user_session'].unique()
    print(f"[Check 3] Bot/Suspicious Sessions dropped: {len(bot_sessions):,} sessions")
    df = df[~df['user_session'].isin(bot_sessions)]

# Check: Cross-session multi-users
# We will keep these as the session is our unit, but we log it for monitoring.
session_user_counts = df.groupby('user_session')['user_id'].nunique()
multi_user_sessions = (session_user_counts > 1).sum()
print(f"[Check 4] Sessions associated with >1 user_id: {multi_user_sessions:,} (Retained)")


EXECUTING SANITY CHECKS & FILTERS
[Check 1] Non-positive prices (<= 0) dropped: 0 rows
[Check 2] Duplicate identical events dropped: 0 rows
[Check 3] Bot/Suspicious Sessions dropped: 0 sessions
[Check 4] Sessions associated with >1 user_id: 114 (Retained)


In [42]:
# SESSION FILTERING FOR PREFIX PREDICTION (N = 3)
N = 3

# Assign sequence order per session
df['event_number'] = df.groupby('user_session').cumcount() + 1

# Identify sessions with >= N events
session_sizes = df.groupby('user_session').size()
valid_length_sessions = session_sizes[session_sizes >= N].index

# Identify sessions with early purchases (purchase <= N)
early_purchases = df[
    (df['event_number'] <= N) & 
    (df['event_type'] == 'purchase')
]['user_session'].unique()

# Filter dataset for modeling
usable_sessions = set(valid_length_sessions) - set(early_purchases)
df_model = df[df['user_session'].isin(usable_sessions)].copy()

print("\n" + "=" * 60)
print("SESSION FILTERING SUMMARY")
print("=" * 60)
print(f"Total Unique Original Sessions: {len(session_sizes):,}")
print(f"Sessions with < {N} events (Dropped): {len(session_sizes) - len(valid_length_sessions):,}")
print(f"Sessions with purchase <= {N} (Dropped): {len(early_purchases):,}")
print(f"Final Valid Modeling Sessions: {df_model['user_session'].nunique():,}")


SESSION FILTERING SUMMARY
Total Unique Original Sessions: 252,064
Sessions with < 3 events (Dropped): 209,591
Sessions with purchase <= 3 (Dropped): 9,053
Final Valid Modeling Sessions: 35,029


In [43]:
print("=" * 60)
print("GENERATING ENRICHED PREFIX FEATURES (N = 3)")
print("=" * 60)

# Slice to include ONLY the first 3 events
prefix_df = df_model[df_model['event_number'] <= N].copy()

# Historical User Context (Leakage-Free)
# Find the start time of EVERY session in the full dataset (df)
session_starts = df.groupby('user_session').agg(
    user_id=('user_id', 'first'),
    session_start=('event_time', 'min')
).reset_index()

# Sort by user and time, then calculate cumulative previous sessions
session_starts = session_starts.sort_values(['user_id', 'session_start'])
session_starts['user_prior_sessions'] = session_starts.groupby('user_id').cumcount()

# Set index for easy joining later
user_history_features = session_starts.set_index('user_session')[['user_prior_sessions']]

GENERATING ENRICHED PREFIX FEATURES (N = 3)


In [44]:
# Behavioral & Price Aggregations
session_features = prefix_df.groupby('user_session').agg(
    view_count=('event_type', lambda x: (x == 'view').sum()),
    cart_count=('event_type', lambda x: (x == 'cart').sum()),
    unique_products=('product_id', 'nunique'),
    unique_categories=('category_id', 'nunique'),
    unique_brands=('brand', 'nunique'),
    average_price=('price', 'mean'),
    max_price=('price', 'max'),
    min_price=('price', 'min')
)

session_features['price_range'] = session_features['max_price'] - session_features['min_price']
session_features['cart_to_view_ratio'] = session_features['cart_count'] / (session_features['view_count'] + 1e-5)

In [45]:
# Exact Time Gaps & Cyclical Temporal Features
# Calculate time gaps between consecutive events
prefix_df['prev_time'] = prefix_df.groupby('user_session')['event_time'].shift(1)
prefix_df['time_gap_sec'] = (prefix_df['event_time'] - prefix_df['prev_time']).dt.total_seconds().fillna(0)

time_gaps = prefix_df.pivot(index='user_session', columns='event_number', values='time_gap_sec').fillna(0)
time_gaps.columns = ['time_gap_0_1', 'time_gap_1_2', 'time_gap_2_3'] 
time_gaps.drop(columns=['time_gap_0_1'], inplace=True) # Event 1 always has 0 gap

# Prediction time features (Timestamp of Event 3)
time_agg = prefix_df.groupby('user_session').agg(
    first_event_time=('event_time', 'min'),
    prediction_time=('event_time', 'max')
)

time_agg['pre_prediction_duration_seconds'] = (
    time_agg['prediction_time'] - time_agg['first_event_time']
).dt.total_seconds()

# Cyclical Time Encoding
hour = time_agg['prediction_time'].dt.hour
weekday = time_agg['prediction_time'].dt.weekday

time_agg['hour_sin'] = np.sin(2 * np.pi * hour / 24)
time_agg['hour_cos'] = np.cos(2 * np.pi * hour / 24)
time_agg['weekday_sin'] = np.sin(2 * np.pi * weekday / 7)
time_agg['weekday_cos'] = np.cos(2 * np.pi * weekday / 7)
time_agg['is_weekend'] = (weekday >= 5).astype(int)

In [46]:
# Recency / State Features at Event N (3rd Event)

last_events = prefix_df[prefix_df['event_number'] == N].set_index('user_session')
recency_features = last_events[[
    'event_type', 'product_id', 'category_id', 'brand', 
    'category_l1', 'category_l2', 'category_l3', 'price_band'
]].add_prefix('last_')

In [47]:
# Combine all engineered features

X_session = (
    session_features
    .join(time_gaps)
    .join(time_agg)
    .join(recency_features)
    .join(user_history_features)
)

print(f"Engineered Feature Matrix Shape : {X_session.shape[0]:,} sessions, {X_session.shape[1]} features")

Engineered Feature Matrix Shape : 35,029 sessions, 29 features


In [48]:
print("\n" + "=" * 60)
print("TARGET CREATION & CLASS IMBALANCE ANALYSIS")
print("=" * 60)

# Compute target using the COMPLETE session log in df_model
targets = df_model.groupby('user_session')['event_type'].apply(
    lambda x: int((x == 'purchase').any())
).rename('target')

# Join features with target
session_df = X_session.join(targets).reset_index()

# Display Target Distribution
print("Target Value Counts (Raw):")
print(session_df['target'].value_counts())

print("\nTarget Distribution (Normalized):")
print(session_df['target'].value_counts(normalize=True).map("{:.2%}".format))


TARGET CREATION & CLASS IMBALANCE ANALYSIS
Target Value Counts (Raw):
target
0    28474
1     6555
Name: count, dtype: int64

Target Distribution (Normalized):
target
0    81.29%
1    18.71%
Name: proportion, dtype: object


In [49]:
print("=" * 60)
print("DATA QUALITY AUDIT")
print("=" * 60)


# Inspect Missing Values
missing_per_col = session_df.isnull().sum()
cols_with_missing = missing_per_col[missing_per_col > 0]
print("Columns with Missing Values:")
if len(cols_with_missing) > 0:
    print(cols_with_missing)
else:
    print("None")

DATA QUALITY AUDIT
Columns with Missing Values:
None


In [50]:
# Inspect Infinite Values
num_cols = session_df.select_dtypes(include=[np.number]).columns
inf_mask = np.isinf(session_df[num_cols])
inf_counts = inf_mask.sum()[inf_mask.sum() > 0]

print("\nColumns with Infinite Values:")

if len(inf_counts) > 0:
    print(inf_counts)
else:
    print("None")

# Check for Duplicate Sessions
duplicate_sessions = session_df['user_session'].duplicated().sum()
print(f"\nDuplicate Session Entries: {duplicate_sessions}")

# Check Constant / Zero-Variance Features
constant_cols = [col for col in session_df.columns if session_df[col].nunique() <= 1]
print(f"\nConstant Features Detected: {constant_cols}")


Columns with Infinite Values:
None

Duplicate Session Entries: 0

Constant Features Detected: []


In [51]:
print("\n" + "=" * 60)
print("CHRONOLOGICAL TIME-BASED DATA SPLIT")
print("=" * 60)

# Sort strictly by prediction_time
session_df = session_df.sort_values('prediction_time').reset_index(drop=True)

# Determine Quantile Cutoffs
train_cutoff = session_df['prediction_time'].quantile(0.70)
val_cutoff = session_df['prediction_time'].quantile(0.85)

# Create Split DataFrames
train_df = session_df[session_df['prediction_time'] <= train_cutoff].copy()
val_df = session_df[
    (session_df['prediction_time'] > train_cutoff) & 
    (session_df['prediction_time'] <= val_cutoff)
].copy()
test_df = session_df[session_df['prediction_time'] > val_cutoff].copy()

# Report Split Metrics
def print_split_info(name, df_subset):
    total = len(df_subset)
    start = df_subset['prediction_time'].min()
    end = df_subset['prediction_time'].max()
    pos_rate = df_subset['target'].mean()
    print(f"{name:<12}: {total:>6,} rows ({total/len(session_df):>5.1%}) | "
          f"Target Mean: {pos_rate:.2%} | "
          f"Range: {start} to {end}")

print_split_info("TRAIN", train_df)
print_split_info("VALIDATION", val_df)
print_split_info("TEST", test_df)


CHRONOLOGICAL TIME-BASED DATA SPLIT
TRAIN       : 24,520 rows (70.0%) | Target Mean: 18.34% | Range: 2020-09-24 12:08:41+00:00 to 2021-01-21 07:53:32+00:00
VALIDATION  :  5,254 rows (15.0%) | Target Mean: 18.31% | Range: 2021-01-21 07:56:06+00:00 to 2021-02-09 02:49:32+00:00
TEST        :  5,255 rows (15.0%) | Target Mean: 20.88% | Range: 2021-02-09 03:13:16+00:00 to 2021-02-28 23:15:04+00:00


In [52]:
print("=" * 60)
print("SEPARATE X AND Y")
print("=" * 60)

drop_cols = ['target', 'user_session', 'first_event_time', 'prediction_time']

# Ensure we drop event_count if it exists (it is a constant N=3)
if 'event_count' in train_df.columns:
    drop_cols.append('event_count')

X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df['target']

X_val = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
y_val = val_df['target']

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_test = test_df['target']

# Keep timestamps for error analysis later
val_times = val_df['prediction_time']
test_times = test_df['prediction_time']

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape  : {X_val.shape} | y_val shape  : {y_val.shape}")
print(f"X_test shape : {X_test.shape} | y_test shape : {y_test.shape}")

SEPARATE X AND Y
X_train shape: (24520, 27) | y_train shape: (24520,)
X_val shape  : (5254, 27) | y_val shape  : (5254,)
X_test shape : (5255, 27) | y_test shape : (5255,)


In [53]:
print("\n" + "=" * 60)
print("FEATURE TYPES & CARDINALITY")
print("=" * 60)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()

# Define cardinality threshold
CARDINALITY_THRESHOLD = 15

high_card_cats = [c for c in cat_cols if X_train[c].nunique() >= CARDINALITY_THRESHOLD]
low_card_cats = [c for c in cat_cols if c not in high_card_cats]

print(f"High-Cardinality Categoricals (>={CARDINALITY_THRESHOLD} unique):")
for c in high_card_cats:
    print(f"  - {c}: {X_train[c].nunique():,} unique values")

print(f"\nLow-Cardinality Categoricals (<{CARDINALITY_THRESHOLD} unique):")
for c in low_card_cats:
    print(f"  - {c}: {X_train[c].nunique():,} unique values")

# Define skewed numericals for log transformation
skewed_num_cols = [
    'average_price', 'max_price', 'min_price', 'price_range', 
    'pre_prediction_duration_seconds', 'time_gap_1_2', 'time_gap_2_3'
]
# Ensure they exist in num_cols
skewed_num_cols = [c for c in skewed_num_cols if c in num_cols]
normal_num_cols = [c for c in num_cols if c not in skewed_num_cols]


FEATURE TYPES & CARDINALITY
High-Cardinality Categoricals (>=15 unique):
  - last_brand: 338 unique values
  - last_category_l2: 15 unique values
  - last_category_l3: 32 unique values

Low-Cardinality Categoricals (<15 unique):
  - last_event_type: 2 unique values
  - last_category_l1: 2 unique values
  - last_price_band: 8 unique values


In [54]:
print("\n" + "=" * 60)
print("PIPELINE CONSTRUCTION & ENCODING")
print("=" * 60)

# Log Transformation Pipeline
log_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

# Standard Numerical Pipeline
std_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Target Encoding Pipeline (For High Cardinality)
# smoothing=10 requires a category to have a decent number of samples before trusting its conversion rate
target_enc_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('target_encode', ce.TargetEncoder(smoothing=10)) 
])

# One-Hot Encoding Pipeline (For Low Cardinality)
ohe_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Assemble Master Preprocessor (FIXED VARIABLE NAMES HERE)
preprocessor = ColumnTransformer(
    transformers=[
        ('log_num', log_transformer, skewed_num_cols),
        ('std_num', std_transformer, normal_num_cols),
        ('target_cat', target_enc_transformer, high_card_cats), 
        ('ohe_cat', ohe_transformer, low_card_cats)
    ],
    remainder='drop'
)

# FIT ONLY ON TRAIN
X_train_proc = preprocessor.fit_transform(X_train, y_train)

# TRANSFORM VAL AND TEST
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print("Preprocessor fitted successfully without leakage.")
print(f"Processed Train Matrix Shape: {X_train_proc.shape}")
print(f"Processed Val Matrix Shape: {X_val_proc.shape}")
print(f"Processed Test Matrix Shape: {X_test_proc.shape}")


PIPELINE CONSTRUCTION & ENCODING
Preprocessor fitted successfully without leakage.
Processed Train Matrix Shape: (24520, 36)
Processed Val Matrix Shape: (5254, 36)
Processed Test Matrix Shape: (5255, 36)


In [55]:
print("=" * 60)
print("MODEL TRAINING & EVALUATION")
print("=" * 60)

# Clean Raw Categorical Columns for Native CatBoost Processing
# CatBoost requires categorical features to be strings without NaNs
for col in cat_cols:
    X_train[col] = X_train[col].fillna('missing').astype(str)
    X_val[col] = X_val[col].fillna('missing').astype(str)
    X_test[col] = X_test[col].fillna('missing').astype(str)

cat_cols_indices = [X_train.columns.get_loc(col) for col in cat_cols]

MODEL TRAINING & EVALUATION


In [56]:
# Train Model 1: Dummy Classifier (Baseline)

dummy_model = DummyClassifier(strategy='prior')
dummy_model.fit(X_train_proc, y_train)
y_val_proba_dummy = dummy_model.predict_proba(X_val_proc)[:, 1]

In [57]:
# Train Model 2: Logistic Regression
lr_model = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000, 
    random_state=42
)
lr_model.fit(X_train_proc, y_train)
y_val_proba_lr = lr_model.predict_proba(X_val_proc)[:, 1]

In [58]:
# Train Model 3: XGBoost Classifier
# Calculate positive class weighting ratio for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_proc, y_train)
y_val_proba_xgb = xgb_model.predict_proba(X_val_proc)[:, 1]

In [59]:
# 5. Train Model 4: CatBoost Classifier

cat_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    loss_function='Logloss',
    eval_metric='PRAUC',
    random_seed=42,
    verbose=0
)

# Fit CatBoost on RAW Data
cat_model.fit(
    X_train, y_train,
    cat_features=cat_cols_indices,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)
y_val_proba_cat = cat_model.predict_proba(X_val)[:, 1]

In [60]:
# 6. Evaluation on Validation Set

def evaluate_predictions(name, y_true, y_proba, threshold=0.5):
    preds = (y_proba >= threshold).astype(int)
    return {
        "Model": name,
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "Precision (@0.5)": precision_score(y_true, preds, zero_division=0),
        "Recall (@0.5)": recall_score(y_true, preds, zero_division=0),
        "F1 (@0.5)": f1_score(y_true, preds, zero_division=0)
    }

comparison_results = [
    evaluate_predictions("Dummy Classifier", y_val, y_val_proba_dummy),
    evaluate_predictions("Logistic Regression", y_val, y_val_proba_lr),
    evaluate_predictions("XGBoost", y_val, y_val_proba_xgb),
    evaluate_predictions("CatBoost", y_val, y_val_proba_cat)
]

comparison_df = pd.DataFrame(comparison_results).sort_values("PR-AUC", ascending=False)

print("\n" + "=" * 60)
print("VALIDATION SET MODEL COMPARISON (DEFAULT THRESHOLD = 0.5)")
print("=" * 60)
print(comparison_df.to_string(index=False))


VALIDATION SET MODEL COMPARISON (DEFAULT THRESHOLD = 0.5)
              Model  ROC-AUC   PR-AUC  Precision (@0.5)  Recall (@0.5)  F1 (@0.5)
           CatBoost 0.752639 0.406316          0.349666       0.652807   0.455402
            XGBoost 0.741994 0.392124          0.344074       0.654886   0.451128
Logistic Regression 0.738582 0.377405          0.356930       0.623701   0.454030
   Dummy Classifier 0.500000 0.183099          0.000000       0.000000   0.000000


In [61]:
print("=" * 60)
print("CATBOOST THRESHOLD OPTIMIZATION (VALIDATION SET)")
print("=" * 60)

# Evaluate thresholds from 0.20 to 0.80
thresholds = np.arange(0.20, 0.85, 0.05)
threshold_results = []

for t in thresholds:
    preds = (y_val_proba_cat >= t).astype(int)
    threshold_results.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_val, preds, zero_division=0),
        "Recall": recall_score(y_val, preds, zero_division=0),
        "F1-Score": f1_score(y_val, preds, zero_division=0)
    })

thresh_df = pd.DataFrame(threshold_results).sort_values("F1-Score", ascending=False)

# Select the optimal threshold based on F1-Score
best_threshold = thresh_df.iloc[0]["Threshold"]
best_f1 = thresh_df.iloc[0]["F1-Score"]

print(thresh_df.head(7).to_string(index=False))
print(f"\nOptimal Decision Threshold Selected: {best_threshold:.2f} (Validation F1 = {best_f1:.4f})")


print("\n" + "=" * 60)
print("TOP 15 FEATURE IMPORTANCES (CATBOOST)")
print("=" * 60)

# 3. Extract Feature Importances
cb_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': cat_model.get_feature_importance()
}).sort_values('Importance', ascending=False)

print(cb_importances.head(15).to_string(index=False))

CATBOOST THRESHOLD OPTIMIZATION (VALIDATION SET)
 Threshold  Precision   Recall  F1-Score
      0.65   0.421603 0.503119  0.458768
      0.60   0.376676 0.584200  0.458028
      0.50   0.349666 0.652807  0.455402
      0.55   0.358254 0.622661  0.454822
      0.45   0.308967 0.755717  0.438612
      0.70   0.451234 0.399168  0.423607
      0.40   0.276420 0.834719  0.415309

Optimal Decision Threshold Selected: 0.65 (Validation F1 = 0.4588)

TOP 15 FEATURE IMPORTANCES (CATBOOST)
                        Feature  Importance
                     view_count   16.386955
             cart_to_view_ratio   11.404154
               last_category_l2   10.408920
               last_category_l3    9.366016
                     cart_count    7.385143
                     last_brand    4.960038
                last_event_type    4.138089
                last_price_band    3.963580
                      min_price    3.919406
                      max_price    3.201476
                unique_products 

In [62]:
print("=" * 60)
print("FINAL EVALUATION ON UNTOUCHED TEST SET")
print("=" * 60)

# Predict probabilities on the UNTOUCHED Test Set
y_test_proba = cat_model.predict_proba(X_test)[:, 1]

# Apply the optimal threshold found in Validation
y_test_pred_optimal = (y_test_proba >= best_threshold).astype(int)

# Calculate core metrics
test_roc = roc_auc_score(y_test, y_test_proba)
test_prauc = average_precision_score(y_test, y_test_proba)

print(f"Test Set Date Range: {test_times.min()} to {test_times.max()}")
print(f"Test ROC-AUC: {test_roc:.4f}")
print(f"Test PR-AUC: {test_prauc:.4f}")

print(f"\n--- Classification Report (Optimal Threshold = {best_threshold:.2f}) ---")
print(classification_report(y_test, y_test_pred_optimal, digits=4))

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_test_pred_optimal)
cm_df = pd.DataFrame(cm, index=['Actual No Purchase', 'Actual Purchase'], 
                         columns=['Pred No Purchase', 'Pred Purchase'])
print(cm_df)

FINAL EVALUATION ON UNTOUCHED TEST SET
Test Set Date Range: 2021-02-09 03:13:16+00:00 to 2021-02-28 23:15:04+00:00
Test ROC-AUC: 0.7605
Test PR-AUC: 0.4679

--- Classification Report (Optimal Threshold = 0.65) ---
              precision    recall  f1-score   support

           0     0.8675    0.8439    0.8555      4158
           1     0.4636    0.5114    0.4863      1097

    accuracy                         0.7745      5255
   macro avg     0.6656    0.6777    0.6709      5255
weighted avg     0.7832    0.7745    0.7785      5255


--- Confusion Matrix ---
                    Pred No Purchase  Pred Purchase
Actual No Purchase              3509            649
Actual Purchase                  536            561


In [63]:
print("=" * 60)
print("CHRONOLOGICAL HYPERPARAMETER TUNING")
print("=" * 60)

# Combine Train and Validation sets for the PredefinedSplit
X_tune = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_tune = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# Create the PredefinedSplit (Train = -1, Val = 0)
train_indices = np.full(len(X_train), -1)
val_indices = np.full(len(X_val), 0)
test_fold = np.concatenate([train_indices, val_indices])
ps = PredefinedSplit(test_fold)

# Define CatBoost parameter grid
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 9],
    'iterations': [300, 500, 800],
    'border_count': [32, 64, 128]
}

# Initialize CatBoost Base Model
cat_tune = CatBoostClassifier(
    auto_class_weights='Balanced',
    loss_function='Logloss',
    random_seed=42,
    verbose=0
)

# Set up RandomizedSearchCV
pr_auc_scorer = make_scorer(average_precision_score, response_method='predict_proba')

random_search = RandomizedSearchCV(
    estimator=cat_tune,
    param_distributions=param_grid,
    n_iter=10,           
    scoring=pr_auc_scorer,
    cv=ps,               
    random_state=42,
    n_jobs=1             
)

# Execute Tuning
print("Executing search... this may take a few minutes depending on compute.")
random_search.fit(
    X_tune, 
    y_tune, 
    cat_features=cat_cols_indices
)

print("\n" + "=" * 60)
print("TUNING RESULTS")
print("=" * 60)
print(f"Best Validation PR-AUC: {random_search.best_score_:.4f}")
print("Best Parameters:")
for k, v in random_search.best_params_.items():
    print(f"  - {k}: {v}")

CHRONOLOGICAL HYPERPARAMETER TUNING
Executing search... this may take a few minutes depending on compute.

TUNING RESULTS
Best Validation PR-AUC: 0.4066
Best Parameters:
  - learning_rate: 0.01
  - l2_leaf_reg: 1
  - iterations: 300
  - depth: 6
  - border_count: 64


In [64]:
# Extract the tuned model
final_model = random_search.best_estimator_

# Re-optimize the threshold for the tuned model on the Validation set
y_val_proba_final = final_model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.20, 0.85, 0.05)
best_thresh, max_f1 = 0.5, 0.0

for t in thresholds:
    f1 = f1_score(y_val, (y_val_proba_final >= t).astype(int), zero_division=0)
    if f1 > max_f1:
        max_f1 = f1
        best_thresh = t

print("=" * 60)
print(f"REFINED OPTIMAL THRESHOLD: {best_thresh:.2f} (Val F1: {max_f1:.4f})")
print("=" * 60)

REFINED OPTIMAL THRESHOLD: 0.60 (Val F1: 0.4669)


In [65]:
# Final Evaluation on the Untouched TEST SET
print("\n" + "=" * 60)
print("FINAL EVALUATION ON UNTOUCHED TEST SET")
print("=" * 60)

y_test_proba_final = final_model.predict_proba(X_test)[:, 1]
y_test_pred_final = (y_test_proba_final >= best_thresh).astype(int)

test_roc = roc_auc_score(y_test, y_test_proba_final)
test_prauc = average_precision_score(y_test, y_test_proba_final)

print(f"Test Set Date Range: {test_times.min()} to {test_times.max()}")
print(f"Test ROC-AUC: {test_roc:.4f}")
print(f"Test PR-AUC: {test_prauc:.4f}")

print(f"\n--- Classification Report (Threshold = {best_thresh:.2f}) ---")
print(classification_report(y_test, y_test_pred_final, digits=4))

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_test_pred_final)
cm_df = pd.DataFrame(cm, index=['Actual No Purchase', 'Actual Purchase'], 
                         columns=['Pred No Purchase', 'Pred Purchase'])
print(cm_df)


FINAL EVALUATION ON UNTOUCHED TEST SET
Test Set Date Range: 2021-02-09 03:13:16+00:00 to 2021-02-28 23:15:04+00:00
Test ROC-AUC: 0.7602
Test PR-AUC: 0.4697

--- Classification Report (Threshold = 0.60) ---
              precision    recall  f1-score   support

           0     0.8860    0.7667    0.8221      4158
           1     0.4146    0.6263    0.4989      1097

    accuracy                         0.7374      5255
   macro avg     0.6503    0.6965    0.6605      5255
weighted avg     0.7876    0.7374    0.7546      5255


--- Confusion Matrix ---
                    Pred No Purchase  Pred Purchase
Actual No Purchase              3188            970
Actual Purchase                  410            687


In [66]:
#

In [78]:
import os
import json

output_dir = "/kaggle/working/"

model_path = os.path.join(output_dir, "catboost_model.cbm")
final_model.save_model(model_path)
print(f"[+] Model saved successfully at: {model_path}")

[+] Model saved successfully at: /kaggle/working/catboost_model.cbm


In [79]:
metadata = {
    "model_name": "CatBoost Intent Predictor (N=3)",
    "optimal_threshold": 0.60,
    "features": list(X_train.columns),
    "categorical_columns": cat_cols
}

metadata_path = os.path.join(output_dir, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"[+] Metadata saved successfully at: {metadata_path}")

[+] Metadata saved successfully at: /kaggle/working/metadata.json
